# 03 · Context and research over MCP

Run top-to-bottom in a fresh project Python kernel. Cells tagged `live` contact external services and may incur model costs. See [source attribution](../ATTRIBUTION.md).

In [1]:
from pathlib import Path
import os, sys, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/accounts_server.py").exists())
sys.path.insert(0, str(ROOT))
# Fresh temporary data for this notebook; never reset the dashboard's accounts.
notebook_data = tempfile.TemporaryDirectory()
os.environ["MCP_DATA_DIR"] = notebook_data.name
os.environ["MARKET_MODE"] = "fixture"
os.environ["ENABLE_PUSH"] = "false"
print("Using isolated fixture data.")

Using isolated fixture data.


## Inspect the context boundaries
Account mutations belong to the trader; web fetching, optional search, and persistent memory belong to the researcher. The researcher is exposed as an agent-as-tool.

In [2]:
import ast
module = ast.parse((ROOT / "backend/mcp_servers.py").read_text())
functions = [node.name for node in module.body if isinstance(node, ast.FunctionDef)]
assert {"trader_mcp_servers", "researcher_mcp_servers"} <= set(functions)
functions

['trader_mcp_servers', 'researcher_mcp_servers']

## Fetch and filesystem tool discovery
These external server packages are downloaded by uvx/npx; the filesystem is restricted to the notebook directory.

In [ ]:
from agents.mcp import MCPServerStdio
from contextlib import AsyncExitStack
params = [
    {"command": "uvx", "args": ["--with", "mcp<2", "mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", notebook_data.name]},
]
async with AsyncExitStack() as stack:
    for configuration in params:
        server = await stack.enter_async_context(MCPServerStdio(params=configuration, client_session_timeout_seconds=120))
        print([tool.name for tool in await server.list_tools()])

## Research with fetch, optional Tavily, and persistent memory

In [ ]:
from examples.research_agent import research
await research("Read https://modelcontextprotocol.io/introduction and summarize the role of tools and resources. Store a concise sourced summary in memory.")

## Optional vector retrieval
Qdrant is a separate integration example, not required by the trading runtime. Its local embedding model may be downloaded on first use.

In [ ]:
vector_path = Path(notebook_data.name) / "qdrant"
configuration = {"command": "uvx", "args": ["mcp-server-qdrant"], "env": {"QDRANT_LOCAL_PATH": str(vector_path), "COLLECTION_NAME": "research"}}
async with MCPServerStdio(params=configuration, client_session_timeout_seconds=120) as server:
    print([tool.name for tool in await server.list_tools()])